[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/02_checkout-ab-test.ipynb)

*Industry Project 2 of 10 · Product / Data Analyst · Intermediate · about 5–7 hours*

# Checkout Redesign: A/B Test Readout

**The situation.** *ShopKart*'s product team redesigned checkout into a single page and ran an experiment: each visitor was randomly assigned to `control` (old checkout) or `new_checkout`. The experiment is over and the product manager asks: **"Should we ship it?"**

A good readout doesn't stop at "p < 0.05". You'll check the experiment was run properly, find where in the funnel the change acted, test significance, size the effect with a confidence interval, check it holds across devices, make sure it didn't harm a guardrail metric (refunds), and turn it into a clear recommendation.

**What you'll hand in:** an experiment readout: validity check, funnel by variant, effect size with a 95% CI and p-value, device breakdown, guardrail check, a ship / don't-ship decision with reasons, and the sample size you'd need for a follow-up test.

**Skills:** A/B testing, funnels, sample ratio mismatch, two-proportion z-test, confidence intervals, segments, guardrail metrics, power analysis

**How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**ab_users.csv** — one row per user in the experiment: user_id, variant (`control` / `new_checkout`), device (`mobile` / `desktop`), country

**ab_events.csv** — funnel steps each user reached: user_id, step (`visit` → `cart` → `checkout` → `purchase`)

**ab_orders.csv** — one row per purchase: user_id, order_value (₹), refunded (1 = refunded)

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return
    print(f"❌ {name}: not yet — {_why(got, want)}")

from scipy import stats
users = load("ab_users.csv")
events = load("ab_events.csv")
ab_orders = load("ab_orders.csv")
STEPS = ["visit", "cart", "checkout", "purchase"]
print(users.variant.value_counts().to_dict())
print("Ready! Data loaded.")

## Milestone 1: Was the split fair? (sample ratio mismatch)

Before reading any result, check the randomisation worked. With a 50/50 split, the two group sizes should differ only by chance. Write `srm(users)` returning a dict with each variant's user count and `p_value` from a chi-square goodness-of-fit test against an equal split (rounded to 4).

A tiny p-value (say < 0.001) means the split is broken and the results can't be trusted.

<details>
<summary><b>💡 Hint</b></summary>

`stats.chisquare(counts)` tests against equal expected counts by default.

</details>

In [ ]:
def srm(users):
    raise NotImplementedError

In [ ]:
check("Was the split fair? (sample ratio mismatch)", lambda: srm(users), {'control': 6042, 'new_checkout': 5958, 'p_value': 0.4432}, tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def srm(users):
    counts = users["variant"].value_counts().sort_index()
    p = stats.chisquare(counts.values).pvalue
    return {**counts.to_dict(), "p_value": round(float(p), 4)}
```

</details>

## Milestone 2: Funnel by variant

Write `funnel(events, users)` returning `{variant: [users at visit, cart, checkout, purchase]}`. Count **distinct users** at each step.

Look at the numbers: at which step does the new checkout differ?

<details>
<summary><b>💡 Hint</b></summary>

Merge events with users to get the variant, then `groupby(['variant', 'step'])['user_id'].nunique()`.

</details>

In [ ]:
def funnel(events, users):
    raise NotImplementedError

In [ ]:
check("Funnel by variant", lambda: funnel(events, users), {'control': [6042, 2265, 1227, 643], 'new_checkout': [5958, 2235, 1255, 724]}, tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def funnel(events, users):
    e = events.merge(users[["user_id", "variant"]], on="user_id")
    counts = e.groupby(["variant", "step"])["user_id"].nunique().unstack()[STEPS]
    return {v: [int(x) for x in row] for v, row in counts.iterrows()}
```

</details>

## Milestone 3: Conversion and lift

The primary metric is **purchase conversion** = users who purchased ÷ users in the variant. Write `conversion(events, users)` returning a dict:
`{"control": rate %, "new_checkout": rate %, "lift_pp": difference in percentage points, "lift_rel_pct": relative lift %}`, all rounded to 2.

<details>
<summary><b>💡 Hint</b></summary>

Users who purchased: `set(events[events.step == 'purchase'].user_id)`; flag them in `users` and take the mean per variant.

</details>

In [ ]:
def conversion(events, users):
    raise NotImplementedError

In [ ]:
check("Conversion and lift", lambda: conversion(events, users), {'control': 10.64, 'new_checkout': 12.15, 'lift_pp': 1.51, 'lift_rel_pct': 14.18}, tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def conversion(events, users):
    buyers = set(events.loc[events["step"] == "purchase", "user_id"])
    rate = users.assign(bought=users["user_id"].isin(buyers)).groupby("variant")["bought"].mean() * 100
    c, t = rate["control"], rate["new_checkout"]
    return {"control": round(c, 2), "new_checkout": round(t, 2), "lift_pp": round(t - c, 2), "lift_rel_pct": round((t - c) / c * 100, 2)}
```

</details>

## Milestone 4: Is it significant? z-test and confidence interval

Write `z_and_ci(events, users)` returning `(z, p_value, ci_low, ci_high)`:
- `z`, `p_value`: two-sided **two-proportion z-test** using the pooled rate (rounded to 4),
- `ci_low`, `ci_high`: 95% confidence interval for (new − control) in **percentage points**, using the unpooled standard error (rounded to 2).

<details>
<summary><b>💡 Hint</b></summary>

Pooled p̂ = all buyers ÷ all users; SE = √(p̂(1−p̂)(1/n₁+1/n₂)). For the CI, SE = √(p₁(1−p₁)/n₁ + p₂(1−p₂)/n₂) and multiply by `stats.norm.ppf(0.975)`.

</details>

In [ ]:
def z_and_ci(events, users):
    raise NotImplementedError

In [ ]:
check("Is it significant? z-test and confidence interval", lambda: z_and_ci(events, users), [2.6024, 0.0093, 0.37, 2.65], tol=0.002)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def z_and_ci(events, users):
    buyers = set(events.loc[events["step"] == "purchase", "user_id"])
    g = users.assign(b=users["user_id"].isin(buyers)).groupby("variant")["b"].agg(["sum", "count"])
    x1, n1 = g.loc["control"]
    x2, n2 = g.loc["new_checkout"]
    p1, p2 = x1 / n1, x2 / n2
    p = (x1 + x2) / (n1 + n2)
    z = (p2 - p1) / math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    pval = 2 * stats.norm.sf(abs(z))
    se = math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    zc = stats.norm.ppf(0.975)
    d = (p2 - p1) * 100
    return (round(z, 4), round(float(pval), 4), round(d - zc * se * 100, 2), round(d + zc * se * 100, 2))
```

</details>

## Milestone 5: Does it hold on every device?

An average can hide opposite effects. Write `lift_by_device(events, users)` returning `{device: lift in percentage points}` (new − control conversion, rounded to 2).

What do you see? What would you tell the team?

In [ ]:
def lift_by_device(events, users):
    raise NotImplementedError

In [ ]:
check("Does it hold on every device?", lambda: lift_by_device(events, users), {'desktop': -0.33, 'mobile': 2.44}, tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def lift_by_device(events, users):
    buyers = set(events.loc[events["step"] == "purchase", "user_id"])
    rate = users.assign(b=users["user_id"].isin(buyers)).groupby(["device", "variant"])["b"].mean() * 100
    return {d: round(rate[d]["new_checkout"] - rate[d]["control"], 2) for d in sorted(users["device"].unique())}
```

</details>

The new page helps mobile a lot and does nothing (or slightly harms) desktop. That's a finding worth more than the overall p-value: you might ship to mobile first and investigate desktop.

## Milestone 6: Guardrail: did refunds go up?

A change that sells more but causes more refunds may not be a win. Write `refund_guardrail(ab_orders, users)` returning `{"control": refund rate %, "new_checkout": refund rate %, "p_value": two-proportion z-test p}` — rates rounded to 2, p to 4. Refund rate = refunded orders ÷ orders.

In [ ]:
def refund_guardrail(ab_orders, users):
    raise NotImplementedError

In [ ]:
check("Guardrail: did refunds go up?", lambda: refund_guardrail(ab_orders, users), {'control': 4.51, 'new_checkout': 4.83, 'p_value': 0.7771}, tol=0.002)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def refund_guardrail(ab_orders, users):
    o = ab_orders.merge(users[["user_id", "variant"]], on="user_id")
    g = o.groupby("variant")["refunded"].agg(["sum", "count"])
    x1, n1 = g.loc["control"]
    x2, n2 = g.loc["new_checkout"]
    p = (x1 + x2) / (n1 + n2)
    z = (x2 / n2 - x1 / n1) / math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    return {"control": round(x1 / n1 * 100, 2), "new_checkout": round(x2 / n2 * 100, 2),
            "p_value": round(float(2 * stats.norm.sf(abs(z))), 4)}
```

</details>

## Milestone 7: Make the call

Write `decide(srm_p, p_value, ci_low, refund_p, refund_up)` that returns one of:
- `"invalid test"` if the split is broken (`srm_p < 0.001`),
- `"don't ship"` if refunds rose significantly (`refund_up` and `refund_p < 0.05`),
- `"ship"` if the result is significant (`p_value < 0.05`) **and** the whole CI is above zero (`ci_low > 0`),
- otherwise `"keep testing"`.

The check runs it on your real numbers and on a few made-up cases.

In [ ]:
def decide(srm_p, p_value, ci_low, refund_p, refund_up):
    raise NotImplementedError

In [ ]:
check("Make the call", lambda: [decide(srm(users)['p_value'], z_and_ci(events, users)[1], z_and_ci(events, users)[2], refund_guardrail(ab_orders, users)['p_value'], refund_guardrail(ab_orders, users)['new_checkout'] > refund_guardrail(ab_orders, users)['control']), decide(0.0001, 0.01, 1, 0.5, False), decide(0.4, 0.01, 0.5, 0.02, True), decide(0.4, 0.2, -0.3, 0.6, False)], ['ship', 'invalid test', "don't ship", 'keep testing'], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def decide(srm_p, p_value, ci_low, refund_p, refund_up):
    if srm_p < 0.001:
        return "invalid test"
    if refund_up and refund_p < 0.05:
        return "don't ship"
    if p_value < 0.05 and ci_low > 0:
        return "ship"
    return "keep testing"
```

</details>

## Milestone 8: Plan the follow-up test

The team wants a desktop-only follow-up. Write `users_needed(base_rate, lift_pp, alpha=0.05, power=0.8)` returning the users needed **per group** to detect an absolute lift of `lift_pp` percentage points from `base_rate` (a fraction), rounded up:

n = (z₁₋α/₂ + z_power)² × (p₁(1−p₁) + p₂(1−p₂)) / (p₂ − p₁)²

In [ ]:
def users_needed(base_rate, lift_pp, alpha=0.05, power=0.8):
    raise NotImplementedError

In [ ]:
check("Plan the follow-up test", lambda: [users_needed(0.13, 1), users_needed(0.13, 2), users_needed(0.13, 1, power=0.9)], [18328, 4722, 24535], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def users_needed(base_rate, lift_pp, alpha=0.05, power=0.8):
    p1, p2 = base_rate, base_rate + lift_pp / 100
    z = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    return math.ceil(z ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2)
```

</details>

Halving the effect you want to detect roughly **quadruples** the users you need. That's why teams agree the minimum effect worth detecting *before* the test.

## A chart for the readout

In [ ]:
import matplotlib.pyplot as plt
e = events.merge(users[["user_id", "variant"]], on="user_id")
f = e.groupby(["step", "variant"])["user_id"].nunique().unstack().loc[STEPS]
(f / f.loc["visit"] * 100).plot(kind="bar", figsize=(7, 3), color=["#94a3b8", "#0f766e"])
plt.ylabel("% of visitors"); plt.title("The new checkout wins at the last step"); plt.xticks(rotation=0); plt.show()

## Present your work

Write the readout as you'd post it to the product team (half a page):

1. **Decision** first: ship / don't ship / keep testing, in one line.
2. **Validity**: the split was fair (SRM p-value).
3. **Result**: conversion control vs new, lift in pp and %, 95% CI, p-value.
4. **Where it acted**: the funnel step, and the device breakdown.
5. **Guardrail**: refunds.
6. **Next steps**: e.g. ship to mobile, follow-up desktop test with N users per group.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Analysed a 12,000-user checkout A/B test end to end: randomisation check, funnel, z-test and 95% CI
- Found a **+X pp** conversion lift driven entirely by mobile, recommending a mobile-first rollout
- Sized a follow-up experiment (**N** users per group for 80% power) and defined a refund guardrail

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. What is a sample ratio mismatch and why check it first?
2. Explain your p-value and confidence interval to a product manager without jargon.
3. The overall lift is positive but desktop is flat. What would you do?
4. Why shouldn't you check the p-value every day and stop when it's significant?
5. How did you decide the experiment needed N users? What happens if you want to detect half the effect?
6. What guardrail metrics would you add for a checkout change?

## Take it further

- Compare average order value between variants with a t-test or bootstrap (it's skewed — which is better?).
- Do a Bayesian analysis: the probability that the new checkout is better, using Beta posteriors.
- Check for a novelty effect: does the lift shrink if you split users by when they arrived?
- Build a reusable `ab_readout()` function that prints the whole report for any experiment.